# KL-Divergenz und Unembedding bei Qwen3.6-35B-A3B: vorbereiteter Test (Version 2)

**Frage.** Hat das Modell eine Unembedding-Anomalie in Bezug auf die Darstellung von KL-Divergenz? Konkret:
1. Schreiben die Neuronen, die mit KL-Divergenz zu tun haben, direkt auf KL-Wörter, und wenn ja, ungewöhnlich stark?
2. Ändert das Ausschalten dieser Neuronen die Vorhersage von KL-Wörtern in KL-Texten stärker als in Kontrolltexten, und stärker als bei Neuronen, die im selben Experten ähnlich aktiv sind?

**Untersucht werden**
- `L24_E124_N345` (positiver Pol im Atlas; KL-Modus mit 24 von 5.000 Feuerungen)
- `L21_E166_N131` (im Atlas nicht katalogisiert; im Label des Atlas der stärkste Treiber des KL-Modus)

**Vier Teile**
0. **Kalibrierung:** Stimmt der Hook mit den gespeicherten Atlas-Aktivierungen überein? (Einheit: Vielfache von θ, wie im Atlas.)
1. **Gewichte (ohne Vorwärtslauf):** direkte Ausgabewirkung auf KL-Wörter, verglichen mit Zufallsdreiergruppen aus dem Vokabular.
2. **Kausal:** Ausschalten an der Zielposition, in KL-Texten und in längenangepassten Kontrolltexten. Das Routing aller Schichten bleibt festgehalten (primär). Kontrollen: Neuronen desselben Experten mit ähnlicher Aktivität.
3. **Numerik:** Untergrenze durch die Batchgröße, No-op-Kontrolle, negative Werte der naiven KL-Formel.

**Vorab festgelegte Regeln** (vor dem Lauf auf dem echten Modell eingefroren)
- **Gate 0:** Von den Atlas-Werten, die zu einer Position mit Wert gehören (Fensterposition ≥ `u`, ohne den Feuer-Token), stimmen mindestens 90 % innerhalb von 0,005 + 5 % (Rundung auf zwei Nachkommastellen) überein, und das Routing stimmt in mindestens 98 % dieser Positionen überein. Sonst bricht das Notebook ab.
- **Gate 0b (nachträglich festgelegt am 28.11.2026, Abweichung vom Plan):** Gate 0 ist im ersten Lauf nicht bestanden. Die Nachrechnung zeigte zwei Ursachen außerhalb des Hooks: 87 % der Atlas-Fenster sind aus längeren Texten geschnitten, und der Atlas hat mit dem Text davor gerechnet. Außerdem ist die Toleranz enger als der Unterschied zwischen zwei Rechenwegen: Auch bei vollem Kontext lagen nur rund 60 % der Werte darin, in float32 auf der CPU wie in bfloat16 auf der A100. Gate 0b prüft deshalb nur die Fenster, die am Textanfang beginnen: Pearson r ≥ 0,99, Steigung durch 0 zwischen 0,95 und 1,05, Routing in mindestens 98 % der Positionen gleich. Mit `GATE0B = True` entscheidet Gate 0b über den Abbruch; Gate 0 wird weiter berechnet und berichtet. Die Schwellen wurden nach Kenntnis der Daten gewählt (Nachrechnung: r = 0,9975, Steigung 0,997, Routing 99,1 %).
- **P1a:** Die KL-Familie gilt als auffällig, wenn ihr oberstes Neuron in der Schicht über dem 99-%-Quantil liegt, das zufällige Dreiergruppen erreichen.
- **P1b:** Die Ausgabevektoren der KL-Wörter gelten als auffällig, wenn ihre Norm im obersten oder untersten Prozent des Vokabulars liegt.
- **P2 (primär, Routing festgehalten):** D = mittlere Änderung der Log-Wahrscheinlichkeit des KL-Wortes minus dieselbe Größe für das Kontrollwort, jeweils gepaart pro Satzpaar, beim Ausschalten an der Zielposition. Gepaarter Bootstrap über Satzpaare; 97,5-%-Intervall (Bonferroni über die zwei Kandidaten). *Spezifisch*, wenn das Intervall ganz unter −0,05 Nat liegt **und** D höchstens so groß ist wie bei allen Kontrollen (Permutations-artiges p ≤ 0,05 mit 19 Kontrollen). *Keine spezifische Wirkung*, wenn das Intervall 0 enthält.
- **P3:** nur berichten, kein Bestehen-Kriterium.

**Offenlegung.** Die Gewichte-Ergebnisse (P1) und die Null-Verteilung wurden vor dem Schreiben dieses Notebooks berechnet. Die P1-Regeln sind danach formuliert. Das ist im Bericht anzugeben.

**Was der Test nicht kann**
- Er zeigt Wirkungen im Modell, nicht ihre Bedeutung für Menschen.
- Das Modell rechnet in BF16. Die gemessene Untergrenze in Teil 3 zeigt, was sich trennen lässt.
- Das schichtweise Laden ist am winzigen Modell gegen das volle Modell geprüft (Gewichte, Logits, Ablationen, No-op, festgehaltenes Routing, Laden aus BF16). Am echten Checkpoint sind nur Namen, Formen und Dtypes geprüft (aus den Dateikopfzeilen). Der erste volle Lauf am echten Modell ist daher Gate 0: Besteht es nicht, bricht das Notebook vor der Auswertung ab, und die Ursache steht in der Ausgabe von Teil 0.
- 30 Satzpaare sind eine kleine Stichprobe; die Konfidenzintervalle und die minimal nachweisbare Wirkung stehen in Teil 2.

**Hardware und Kosten (ungeprüft)**
- Teil 1 braucht das Modell nicht: Mit `P1_FROM_FILES = True` werden nur die nötigen Tensoren (rund 3 GB) geladen; Teile 0, 2 und 3 entfallen dann.
- Teile 0 bis 3 laufen mit schichtweisem Laden auf einer GPU mit mindestens 40 GB (eine A100 mit 40 GB genügt). Die Schichten ab 21 bleiben dauerhaft auf der GPU (rund 32 GiB). Für die Vorberechnung wird jede untere Schicht je Aufruf einmal geladen und rechnet alle Texte. Der Download braucht rund 72 GB Festplattenplatz; das Notebook prüft das vorab.
- Laufzeit nicht gemessen. Schätzung: 2 bis 4 Stunden (rund 5.000 Modellläufe für Teil 2); ein Probelauf (`SMOKE = True`) etwa 15 bis 30 Minuten.
- Atlas-Dateien werden über das Interface-Paket mit der Standardrate geholt (4 Anfragen pro Sekunde). Nichts davon wird gespeichert oder weitergegeben.

In [ ]:
# In Colab einmal ausführen (lokal überspringen, wenn die Pakete vorhanden sind)
!pip install -q "transformers==5.24.1" "tokenizers" "safetensors" "accelerate"
!pip install -q "neuron-atlas-interface @ git+https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI.git@main"
# Optional, beschleunigt die Linear-Attention-Schichten (ohne diese läuft die langsame Referenz-Implementierung):
!pip install -q flash-linear-attention

In [ ]:
# ---------- Konfiguration: vor dem Lauf festlegen ----------
import json, math, random, time
import torch

MODEL_ID = "Qwen/Qwen3.6-35B-A3B"       # BF16, rund 70 GB. Die FP8-Fassung ist ungeprüft.
TEST_MODE = False                        # True nur zum Prüfen des Notebooks (winziges Modell, vom Testlauf bereitgestellt)
SMOKE = False                            # True: wenige Texte und Kontrollen, schneller Probelauf
P1_FROM_FILES = False                    # True: nur Teil 1, ohne Modell (nur die nötigen Tensoren laden)
CALIB_POLE = "L24_E124_N345__pos"        # Atlas-Pol mit gespeicherten Aktivierungen (Kalibrierung)
CANDIDATES = [(24, 124, 345), (21, 166, 131)]   # (Schicht, Experte, Neuron)
W_WINDOW = 16                            # Atlas: Schaden = Summe über 16 Positionen
BATCH = 48                               # gleiche Batchform für Referenz und Ablation
SPLIT_LAYER = 21                         # obere Schichten ab hier dauerhaft auf der GPU (höchstens die kleinste Kandidatenschicht)
STREAM_CHUNK = 64                        # Texte je Block der Vorberechnung (die unteren Schichten werden je Aufruf einmal geladen)
MIN_GPU_GB = 38                          # A100 mit 40 GB; die oberen Schichten brauchen rund 32 GiB
N_MATCH = 19                             # aktivitätsgleiche Kontroll-Neuronen je Kandidat (gleicher Experte)
CAL_ABS, CAL_REL = 0.005, 0.05           # Toleranz Gate 0 (Rundung auf 2 Nachkommastellen + 5 %)
GATE_VALUE, GATE_ROUTE = 0.90, 0.98      # Schwellen Gate 0
GATE0B = True                            # nachträglich (28.11.2026): Gate 0b entscheidet über den Abbruch (Abweichung vom Plan)
GATE0B_R, GATE0B_SLOPE, GATE0B_ROUTE = 0.99, (0.95, 1.05), 0.98   # Gate 0b, nur Fenster ab Textanfang
SEED = 20261127
STOP_IF_CALIBRATION_FAILS = True
FAM_KL = [" divergence", " KL", "KL"]                        # einteilige Zielwörter
FAM_CTL = [" distance", " error", " norm", " Distance"]     # Kontrollwörter
random.seed(SEED)
torch.manual_seed(SEED)
if SMOKE:
    N_MATCH = 2
print("Konfiguration:", dict(MODEL_ID=MODEL_ID, TEST_MODE=TEST_MODE, SMOKE=SMOKE, P1_FROM_FILES=P1_FROM_FILES, CANDIDATES=CANDIDATES))

In [ ]:
# ---------- Bibliothek (getestet an einem winzigen Modell derselben Architektur) ----------
"""Kernfunktionen für den Test: ein einzelnes Neuron eines MoE-Experten ausschalten und die Wirkung messen.

Geschrieben für transformers 5.24.1, Modelltyp qwen3_5_moe (Qwen3.6-35B-A3B). Alle Zugriffe auf
Modulnamen stehen hier an einer Stelle; weichen sie ab, bricht der Code mit einer klaren Meldung ab.

Routing: RoutingControl steuert die Expertenwahl ALLER MoE-Schichten. "free" lässt sie unverändert,
"record" merkt sich die Wahl einer Referenz, "freeze" erzwingt die gemerkte Wahl (und Gewichte) in allen
Schichten. Nur so ist ausgeschlossen, dass das Ausschalten über Routing-Umschaltungen in späteren
Schichten wirkt.
"""
import math
import time
import traceback

import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open


LIB_VERSION = "2026-11-28"   # geht in die Signatur fertiger Läufe ein


# ----------------------------------------------------------------------------- Modulzugriff
def text_model(model):
    """Der Textteil (Embedding, Decoder-Schichten, Endnorm) des Modells."""
    for path in ("model.language_model", "model"):
        m = model
        for part in path.split("."):
            m = getattr(m, part, None)
            if m is None:
                break
        if m is not None and hasattr(m, "layers") and hasattr(m, "embed_tokens"):
            return m
    raise RuntimeError("Textteil nicht gefunden: Modulnamen weichen ab (geprüft mit transformers 5.24.1).")


def moe_block(model, layer):
    blk = text_model(model).layers[layer]
    if isinstance(blk, LazyLayer):
        raise RuntimeError(f"Schicht {layer} liegt in der gestreamten Zone (unter SPLIT): Eingriffe sind nur ab SPLIT möglich.")
    mlp = blk.mlp
    for attr in ("gate", "experts", "shared_expert", "shared_expert_gate"):
        if not hasattr(mlp, attr):
            raise RuntimeError(f"Schicht {layer}: MoE-Block ohne '{attr}' – der Aufbau weicht ab.")
    return mlp


def expert_mats(mlp, e):
    """Gate-, Up- und Down-Matrix des Experten e: gate [I,H], up [I,H], down [H,I].
    Im Modell liegen sie fusioniert: gate_up_proj[e] = [2I,H], erste I Zeilen = Gate, letzte I = Up."""
    down = mlp.experts.down_proj[e]
    I = down.shape[1]
    gu = mlp.experts.gate_up_proj[e]
    return gu[:I], gu[I:], down


def route(mlp, x):
    """Top-k-Routing genau wie im Modell: Softmax über alle Experten, Top-k, dann normiert."""
    logits = F.linear(x, mlp.gate.weight)
    probs = torch.softmax(logits.float(), dim=-1)
    v, i = torch.topk(probs, mlp.gate.top_k, dim=-1)
    return i, v / v.sum(-1, keepdim=True)


def neuron_act(mlp, e, n, x):
    """Aktivierung SiLU(Gate_n) * Up_n des Neurons n im Experten e, für jede Zeile von x (fp32)."""
    g, u, _ = expert_mats(mlp, e)
    gate = x.float() @ g[n].float()
    up = x.float() @ u[n].float()
    return mlp.experts.act_fn(gate) * up


def expert_acts_all(mlp, e, x):
    """Aktivierungen aller I Neuronen des Experten e für jede Zeile von x: [N, I] (fp32)."""
    g, u, _ = expert_mats(mlp, e)
    return mlp.experts.act_fn(x.float() @ g.float().T) * (x.float() @ u.float().T)


def moe_manual(mlp, x, zero_neuron=None):
    """Der MoE-Block nachgebaut aus seinen Gewichten (fp32). zero_neuron=(e,n) schaltet ein Neuron aus."""
    x = x.float()
    top_i, top_v = route(mlp, x)
    out = torch.zeros_like(x)
    for e in torch.unique(top_i).tolist():
        rows, slot = torch.where(top_i == e)
        g, u, d = expert_mats(mlp, e)
        g, u, d = g.float(), u.float(), d.float()
        h = mlp.experts.act_fn(x[rows] @ g.T) * (x[rows] @ u.T)
        if zero_neuron is not None and zero_neuron[0] == e:
            h[:, zero_neuron[1]] = 0.0
        out.index_add_(0, rows, (h @ d.T) * top_v[rows, slot, None])
    shared = torch.sigmoid(x @ mlp.shared_expert_gate.weight.float().T) * mlp.shared_expert(x)
    return out + shared


# ----------------------------------------------------------------------------- Routing
class RoutingControl:
    """Steuert die Expertenwahl aller MoE-Schichten. modes: 'free', 'record', 'freeze'."""

    def __init__(self, model):
        self.mode = "free"
        self.ref = {}        # Schicht -> (Gewichte, Indizes) der Referenz (Batchform muss gleich bleiben)
        self.last = {}       # Schicht -> (Gewichte, Indizes), so wie der Block sie benutzt hat
        self._h = []
        for L, blk in enumerate(text_model(model).layers):
            if hasattr(blk, "mlp") and hasattr(blk.mlp, "gate"):
                self._h.append(blk.mlp.gate.register_forward_hook(self._make(L)))

    def _make(self, L):
        def hook(mod, args, out):
            logits, scores, idx = out
            if self.mode == "freeze" and L in self.ref:
                rs, ri = self.ref[L]
                if rs.shape != scores.shape or ri.shape != idx.shape:
                    raise RuntimeError(f"Schicht {L}: Batchform der Referenz passt nicht ({tuple(ri.shape)} gegen {tuple(idx.shape)})")
                scores, idx = rs.to(scores.device, scores.dtype), ri.to(idx.device)
                out = (logits, scores, idx)
            if self.mode == "record":
                self.ref[L] = (scores.detach().clone(), idx.detach().clone())
            self.last[L] = (scores, idx)
            return out
        return hook

    def remove(self):
        for h in self._h:
            h.remove()

    def reset(self):
        """Grundzustand: freies Routing, keine Referenz (zum Beispiel nach einem abgebrochenen Lauf)."""
        self.mode = "free"
        self.ref.clear()


# Angemeldete NeuronHook- und BlockInput-Objekte. Die Liste bleibt erhalten, wenn diese Zelle erneut ausgeführt
# wird; sonst wäre ein hängen gebliebener Hook danach nicht mehr auffindbar.
_LIVE_HOOKS = globals().get("_LIVE_HOOKS", [])


def remove_neuron_hooks(model=None):
    """Entfernt alle noch angemeldeten NeuronHook- und BlockInput-Hooks und liefert ihre Anzahl. Mit model werden
    zusätzlich alle solchen Hooks am Modell gesucht, auch aus einer früheren Fassung dieser Bibliothek, die kein
    Verzeichnis führte. Nach einem abgebrochenen Lauf hängt ein Hook sonst weiter am Modell."""
    n = len(_LIVE_HOOKS)
    for h in list(_LIVE_HOOKS):
        h.remove()
    if model is not None:
        n += _remove_model_hooks(model, ("NeuronHook.", "BlockInput."))
    return n


def remove_routing_hooks(model):
    """Entfernt die Routing-Hooks aller RoutingControl-Objekte am Modell, auch aus einer früheren Fassung."""
    return _remove_model_hooks(model, ("RoutingControl.",))


def _remove_model_hooks(model, prefixes):
    """Entfernt an MoE-Blöcken und Routern die Hooks, deren Funktion zu einer der genannten Klassen gehört."""
    n = 0
    for blk in text_model(model).layers:
        mlp = getattr(blk, "mlp", None)
        for mod in (mlp, getattr(mlp, "gate", None)):
            if mod is None:
                continue
            for d in (mod._forward_hooks, mod._forward_pre_hooks):
                for k, fn in list(d.items()):
                    if getattr(getattr(fn, "__func__", fn), "__qualname__", "").startswith(prefixes):
                        del d[k]
                        n += 1
    return n


class BlockInput:
    """Merkt sich die Eingabe (normierte Zwischenwerte) des MoE-Blocks einer Schicht."""

    def __init__(self, model, layer):
        self.x = None
        self._h = moe_block(model, layer).register_forward_pre_hook(self._pre)
        _LIVE_HOOKS.append(self)

    def _pre(self, mod, args):
        self.x = args[0].reshape(-1, args[0].shape[-1]).detach()

    def remove(self):
        self._h.remove()
        if self in _LIVE_HOOKS:
            _LIVE_HOOKS.remove(self)


# ----------------------------------------------------------------------------- Hook
class NeuronHook:
    """Zeichnet die Aktivierung eines Neurons (Experte e, Neuron n, Schicht L) auf und/oder schaltet es aus.

    scale = 1.0 rechnet den Eingriff ohne Änderung durch (No-op-Test); scale = 0.0 schaltet das Neuron aus.
    Ausgeschaltet wird nur an den Stellen, die `mask` (bool [B,T]) angibt. Das Routing kommt aus
    RoutingControl.last (so wie der Block es benutzt hat).
    """

    def __init__(self, model, layer, e, n, routing):
        self.mlp = moe_block(model, layer)
        self.layer, self.e, self.n = layer, e, n
        self.routing = routing
        self.mask, self.scale = None, 0.0
        self.record, self.rec = False, {}
        self._h = [self.mlp.register_forward_pre_hook(self._pre),
                   self.mlp.register_forward_hook(self._post)]
        _LIVE_HOOKS.append(self)

    def remove(self):
        for h in self._h:
            h.remove()
        if self in _LIVE_HOOKS:
            _LIVE_HOOKS.remove(self)

    def _pre(self, mod, args):
        self._x = args[0]

    def _post(self, mod, args, out):
        B, T, H = self._x.shape
        x = self._x.reshape(-1, H)
        scores, idx = self.routing.last[self.layer]
        w = (scores * (idx == self.e)).sum(-1).float()           # Routing-Gewicht von e pro Token
        act = neuron_act(self.mlp, self.e, self.n, x)            # Aktivierung des Neurons pro Token
        if self.record:
            self.rec["act"] = act.reshape(B, T).detach().cpu()
            self.rec["w"] = w.reshape(B, T).detach().cpu()
        if self.mask is None:
            return out
        _, _, down = expert_mats(self.mlp, self.e)
        col = down[:, self.n].float()                            # Schreibvektor des Neurons, [H]
        coef = (1.0 - self.scale) * w * act                      # Anteil, der wegfällt (0 bei scale = 1)
        delta = (coef[:, None] * col[None, :]).reshape(B, T, H)
        m = self.mask.to(delta.device).unsqueeze(-1).to(delta.dtype)
        return out - (delta * m).to(out.dtype)


# ----------------------------------------------------------------------------- Läufe
@torch.no_grad()
def logits_of(model, ids):
    """ids [B,T] -> Logits [B,T,V]."""
    return model(input_ids=ids, use_cache=False).logits


@torch.no_grad()
def activation_at(model, hook, routing, ids):
    """Aktivierung und Routing-Gewicht des Neurons für jede Position eines einzelnen Textes (ids [T])."""
    routing.mode = "free"
    hook.mask, hook.scale, hook.record = None, 0.0, True
    try:
        model(input_ids=ids[None], use_cache=False)
    finally:
        hook.record = False
    return hook.rec["act"][0], hook.rec["w"][0]


@torch.no_grad()
def ablation_table(model, hook, routing, ids, target_ids, W=16, batch=32, freeze=None, scale=0.0):
    """Für einen Text (ids [T]) wird Zeile i mit ausgeschaltetem Neuron an Position i gerechnet.

    Referenz (Clean) und alle ausgeschalteten Läufe haben dieselbe Batchform (batch Zeilen).
    freeze='clean': das Routing aller Schichten wird aus der Referenz übernommen.
    target_ids[p] = Token nach Position p (-1 am Textende).
    Liefert:
      kl[i,p]     KL(clean_p || abl_i,p), 0 für p < i (kausal)
      lp_tgt[i,p] log-Wahrscheinlichkeit des Ziel-Tokens an Position p, Zeile i (float64)
      clean_tgt[p] dieselbe Größe im Clean-Lauf (float64)
      act[p], w[p] Aktivierung und Routing-Gewicht des Neurons im Clean-Lauf
      dmg[i]      Summe von kl[i,p] über p = i .. i+W-1 (beschreibend)
    """
    T = ids.shape[0]
    dev = ids.device
    B = max(int(batch), 1)
    full = ids[None].expand(B, -1).contiguous()
    kl = torch.zeros(T, T, dtype=torch.float64)
    lp_tgt = torch.zeros(T, T, dtype=torch.float64)
    try:
        routing.mode = "record"
        hook.mask, hook.scale, hook.record = None, 0.0, True
        clean_logits = logits_of(model, full)[0]                 # Zeile 0 = Referenz in dieser Batchform
        hook.record = False
        act, w = hook.rec["act"][0], hook.rec["w"][0]
        clean_lp = torch.log_softmax(clean_logits.double(), -1)  # float64 gegen Rundungsartefakte erster Ordnung
        tg = target_ids.clamp(min=0).to(dev)
        arangeT = torch.arange(T, device=dev)
        clean_tgt = clean_lp[arangeT, tg].cpu()
        p_clean = clean_lp.exp()
        routing.mode = "freeze" if freeze == "clean" else "free"
        for s in range(0, T, B):
            rows = list(range(s, min(T, s + B)))
            m = torch.zeros(B, T, dtype=torch.bool, device=dev)
            for b, i in enumerate(rows):
                m[b, i] = True
            hook.mask, hook.scale = m, scale
            lg = logits_of(model, full)
            for b, i in enumerate(rows):
                lp_b = torch.log_softmax(lg[b].double(), -1)
                kl[i] = ((p_clean * (clean_lp - lp_b)).sum(-1)).cpu()
                lp_tgt[i] = lp_b[arangeT, tg].cpu()
            del lg
    finally:                                                     # auch nach einem Abbruch bleibt kein Eingriff aktiv
        routing.mode = "free"
        hook.mask, hook.scale, hook.record = None, 0.0, False
    dmg = torch.stack([kl[i, i:min(i + W, T)].sum() for i in range(T)])
    return dict(kl=kl, lp_tgt=lp_tgt, clean_tgt=clean_tgt, act=act, w=w, dmg=dmg)


def term_positions(ids, family):
    """Positionen p, an denen das nächste Token zur Familie gehört (ids[p+1] in family)."""
    fam = set(family)
    return [p for p in range(len(ids) - 1) if ids[p + 1] in fam]


# ----------------------------------------------------------------------------- Gewichte (ohne Vorwärtslauf)
@torch.no_grad()
def direct_effect_columns(W_U, gain, Wd_all, family_rows):
    """Direkte Ausgabewirkung aller Neuronen einer Schicht auf das Vokabular, ohne spätere Schichten.

    W_U [V,H] Ausgabe-Matrix (nur echte Token), gain [H] = 1 + Gewicht der Endnorm, Wd_all [E,H,I] Down-Matrizen,
    family_rows Liste von Token-IDs. Liefert für jedes der E*I Neuronen (Index e*I+n):
      z_fam[c]  Mittel über die Familie von (Wirkung - Mittel über Vokabular) / Standardabweichung über Vokabular
      sd[c]     Standardabweichung der Wirkung über das Vokabular
      mean[c]   Mittel der Wirkung über das Vokabular
    """
    W = W_U.float()
    mu = W.mean(0)
    Wc = W - mu
    C = (Wc.T @ Wc) / W.shape[0]
    WK = W[family_rows] if family_rows else None
    E, H, I = Wd_all.shape
    z_fam = torch.empty(E * I)
    sd = torch.empty(E * I)
    mean_all = torch.empty(E * I)
    for e in range(E):
        M = (gain.float()[:, None] * Wd_all[e].float())
        mean_c = mu @ M
        var_c = (M * (C @ M)).sum(0)
        s_c = var_c.clamp_min(1e-30).sqrt()
        sd[e * I:(e + 1) * I] = s_c
        mean_all[e * I:(e + 1) * I] = mean_c
        if WK is not None:
            zk = ((WK @ M) - mean_c) / s_c
            z_fam[e * I:(e + 1) * I] = zk.mean(0)
    return z_fam, sd, mean_all


@torch.no_grad()
def direct_effect_vector(W_U, gain, Wd_e_n):
    """Wirkung eines einzelnen Neurons auf alle Vokabel-Token: W_U @ (gain * Schreibvektor)."""
    return W_U.float() @ (gain.float() * Wd_e_n.float())


@torch.no_grad()
def family_max_null(W_U, gain, Wd, rows_list):
    """Für jede Wortfamilie (rows_list [F,k] mit Token-IDs) das oberste Neuron der Schicht:
    Wert = Mittel der z-Werte der Familie über das Vokabular, Maximum über alle Neuronen.
    Liefert (Maximalwerte [F], Index e*I+n der Neuronen [F])."""
    W = W_U.float()
    mu = W.mean(0)
    Wc = W - mu
    C = (Wc.T @ Wc) / W.shape[0]
    E, H, I = Wd.shape
    F_, k = rows_list.shape
    Rows = W[rows_list.reshape(-1)]
    best = torch.full((F_,), -1e9)
    arg = torch.zeros(F_, dtype=torch.long)
    for e in range(E):
        M = gain.float()[:, None] * Wd[e].float()
        mean_c = mu @ M
        sd_c = (M * (C @ M)).sum(0).clamp_min(1e-30).sqrt()
        z = (((Rows @ M) - mean_c) / sd_c).reshape(F_, k, I).mean(1)
        vals, idxs = z.max(dim=1)
        upd = vals > best
        best = torch.where(upd, vals, best)
        arg = torch.where(upd, e * I + idxs, arg)
    return best, arg


def percentile_rank(values, x):
    """Anteil der Werte in values, die kleiner oder gleich x sind (0..1)."""
    v = torch.as_tensor(values).float().flatten()
    return float((v <= x).float().mean())


# ----------------------------------------------------------------------------- Statistik
def paired_bootstrap(d_by_pair, n_boot=5000, seed=1, lo_q=0.0125, hi_q=0.9875):
    """Mittelwert der Paardifferenzen mit Bootstrap über Paare. d_by_pair: Liste je Paar (Liste von Werten).
    Gibt (Schätzer, untere Grenze, obere Grenze) zum Niveau hi_q - lo_q."""
    import random
    per = [sum(v) / len(v) for v in d_by_pair if len(v)]
    est = sum(per) / len(per)
    rng = random.Random(seed)
    n = len(per)
    boots = []
    for _ in range(n_boot):
        boots.append(sum(per[rng.randrange(n)] for _ in range(n)) / n)
    boots.sort()
    return est, boots[int(lo_q * n_boot)], boots[min(n_boot - 1, int(hi_q * n_boot))]


def calibration_fit(rows, crop_u=84):
    """Gate 0b (nachträglich festgelegt am 28.11.2026): Übereinstimmung nur auf Fenstern, die am Textanfang beginnen.

    Der Atlas zeigt je Feuerung bis zu 100 Token davor; abgeschnittene Fenster haben u = crop_u und lassen sich ohne den
    Text davor nicht nachrechnen. rows: Einträge mit a (Modell, in theta), v (Atlas-Wert oder None), routed (Modell
    wählt den Experten), pos (j - u) und left (j). Liefert n_slots, n_valued, pearson_r, slope_through_origin und
    share_routing_ok (None, wo nicht berechenbar)."""
    sel = [r for r in rows if r["left"] - r["pos"] < crop_u]
    val = [r for r in sel if r["v"] is not None]
    out = {"n_slots": len(sel), "n_valued": len(val), "pearson_r": None, "slope_through_origin": None,
           "share_routing_ok": None}
    if sel:
        out["share_routing_ok"] = sum(r["routed"] == (r["v"] is not None) for r in sel) / len(sel)
    if len(val) > 2:
        A = torch.tensor([r["a"] for r in val], dtype=torch.float64)
        V = torch.tensor([r["v"] for r in val], dtype=torch.float64)
        r_ = float(torch.corrcoef(torch.stack([A, V]))[0, 1])
        s_ = float((A * V).sum() / (V * V).sum().clamp_min(1e-30))
        out["pearson_r"] = r_ if math.isfinite(r_) else None
        out["slope_through_origin"] = s_ if math.isfinite(s_) else None
    return out


def calibration_b_ok(fit, r_min=0.99, slope=(0.95, 1.05), route_min=0.98):
    """Entscheidung von Gate 0b aus calibration_fit."""
    return bool(fit["pearson_r"] is not None and fit["slope_through_origin"] is not None
                and fit["share_routing_ok"] is not None and fit["pearson_r"] >= r_min
                and slope[0] <= fit["slope_through_origin"] <= slope[1] and fit["share_routing_ok"] >= route_min)


def naive_kl_f32(logits_p, logits_q):
    """KL(p||q) mit dem naiven Ansatz in float32: Wahrscheinlichkeiten per Softmax, dann Summe p*(log p - log q)."""
    p = torch.softmax(logits_p.float(), -1)
    q = torch.softmax(logits_q.float(), -1)
    return (p * (torch.log(p) - torch.log(q))).sum(-1)


def stable_kl_f64(logits_p, logits_q):
    """KL(p||q) aus den Logits in float64 mit log_softmax (ohne Auslöschung)."""
    lp = torch.log_softmax(logits_p.double(), -1)
    lq = torch.log_softmax(logits_q.double(), -1)
    return (lp.exp() * (lp - lq)).sum(-1)


# ----------------------------------------------------------------------------- Safetensors per HTTP-Bereich
def read_safetensors_tensors(url, names):
    """Lädt einzelne Tensoren aus einer .safetensors-Datei über HTTP-Bereichsanfragen (ohne die ganze Datei)."""
    import json
    import urllib.request

    def get(a, b):
        req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
        with urllib.request.urlopen(req, timeout=600) as r:
            return r.read()

    n = int.from_bytes(get(0, 7), "little")
    header = json.loads(get(8, 8 + n - 1))
    base = 8 + n
    out = {}
    dtypes = {"BF16": torch.bfloat16, "F32": torch.float32, "F16": torch.float16, "F8_E4M3": torch.float8_e4m3fn}
    for name in names:
        info = header[name]
        a, b = info["data_offsets"]
        buf = bytearray(get(base + a, base + b - 1))
        t = torch.frombuffer(buf, dtype=dtypes[info["dtype"]]).reshape(info["shape"]).clone()
        out[name] = t
    return out


# ----------------------------------------------------------------------------- Schichtweises Laden
# Das BF16-Modell (rund 70 GB) passt nicht auf eine A100 mit 40 GB. Die oberen Schichten (ab SPLIT, Standard 21)
# bleiben dauerhaft auf der GPU (rund 32 GiB). Die unteren Schichten werden für eine Gruppe von Texten einmal
# nacheinander geladen; ihre Ausgabe, also die Eingabe der Schicht SPLIT, wird pro Text gespeichert. Danach
# brauchen alle Läufe nur noch die oberen Schichten.
# Gerechnet wird mit dem Modellcode von transformers. Untere Schichten und Embedding sind Platzhalter. Für die
# Vorberechnung zeichnen die Platzhalter auf, womit das Modell jede untere Schicht aufruft; danach wird jede
# untere Schicht einmal geladen und rechnet alle Textblöcke. In den Läufen reichen die Platzhalter die
# gespeicherte Eingabe durch. Die Ergebnisse sind dieselben wie im vollen Modell (tiny_stream_test.py).


class _StopAfterSplit(Exception):
    """Bricht die Vorberechnung ab, sobald die Eingabe der Schicht SPLIT vorliegt."""


def progress(iterable, desc, show=True):
    """Fortschrittsanzeige, falls gewünscht und tqdm vorhanden ist."""
    if not show:
        return iterable
    try:
        from tqdm.auto import tqdm
        return tqdm(iterable, desc=desc)
    except Exception:
        return iterable


class LazyLayer(nn.Module):
    """Platzhalter für eine untere Schicht. mode 'pass': durchreichen. mode 'capture': durchreichen und merken,
    womit das Modell die Schicht aufruft. mode 'stream': Gewichte laden, rechnen, wieder freigeben."""

    def __init__(self, streamer, idx):
        super().__init__()
        self.streamer = streamer
        self.idx = idx
        self.mode = "pass"

    def forward(self, hidden_states, **kwargs):
        if self.mode == "pass":
            return hidden_states
        if self.mode == "capture":
            self.streamer._capture(self.idx, hidden_states, kwargs)
            return hidden_states
        layer = self.streamer.build_layer(self.idx)
        out = layer(hidden_states, **kwargs)
        del layer
        return out


class StreamEmbed(nn.Module):
    """Platzhalter für das Embedding. mode 'embed': echtes Embedding. mode 'inject': gespeicherte Eingabe."""

    def __init__(self, real, streamer):
        super().__init__()
        self.real = real
        self.streamer = streamer
        self.mode = "embed"

    def forward(self, input_ids):
        if self.mode == "embed":
            return self.real(input_ids)
        return self.streamer.injected(input_ids)


def skeleton(config):
    """Das vollständige Modell ohne Gewichte (Meta-Gerät). Liefert Namen, Formen und Aufbau."""
    from accelerate import init_empty_weights
    from transformers.models.qwen3_5_moe.modeling_qwen3_5_moe import Qwen3_5MoeForConditionalGeneration
    with init_empty_weights():
        m = Qwen3_5MoeForConditionalGeneration(config)
    return m.eval()


def needed_names(skel):
    """Namen der Gewichte, die Textteil und Ausgabe brauchen (ohne Bildmodul und Mehrfach-Vorhersage)."""
    return [n for n, _ in skel.named_parameters() if n.startswith(("model.language_model.", "lm_head."))]


class LayerStreamer:
    """Lädt Gewichte aus lokalen Dateien, hält die oberen Schichten auf dem Gerät und rechnet die unteren
    Schichten für Texte vor.

    skel:     Modell auf dem Meta-Gerät (skeleton(config)).
    shard_of: Gewichtsname -> lokaler Pfad der .safetensors-Datei, die ihn enthält.
    split:    erste Schicht, die dauerhaft auf dem Gerät liegt (Eingriffe nur ab hier).
    chunk:    Anzahl Texte je Block (ein Block wird gemeinsam gerechnet).

    Einstellbar nach dem Anlegen:
    layer_major   True (Standard): jede untere Schicht wird je ensure-Aufruf einmal geladen und rechnet alle
                  Blöcke. False: jeder Block lädt die unteren Schichten selbst (ursprünglicher Weg).
    state_budget  Bytes an Zwischenzuständen, die ein Durchlauf auf dem Gerät halten darf (Standard 2 GiB).
    verbose       Fortschritt und Zeiten ausgeben.
    """

    def __init__(self, skel, shard_of, device, dtype, split, chunk=64):
        self.layer_major = True
        self.state_budget = 2 * 2**30
        self.verbose = True
        self.stats = {"layer_loads": 0, "texts": 0}     # geladene untere Schichten, vorberechnete Texte
        self._rec = None
        self.model = skel
        self.text = text_model(skel)
        self.tc = skel.config.text_config
        self.shard_of = shard_of
        dev = torch.device(device)
        if dev.type == "cuda" and dev.index is None:
            dev = torch.device("cuda", torch.cuda.current_device())
        self.device = dev
        self.dtype = dtype
        self.split = split
        self.chunk = chunk
        self.pad_id = 0
        self.cache = {}     # Tupel der Token-IDs -> Eingabe der Schicht SPLIT, Form [1, T, H]
        n = len(self.text.layers)
        if not (1 <= split < n):
            raise ValueError(f"split muss zwischen 1 und {n - 1} liegen, ist {split}")
        self.embed = StreamEmbed(self.text.embed_tokens, self)
        self.text.embed_tokens = self.embed
        self.lazy = []
        for i in range(split):
            lz = LazyLayer(self, i)
            self.text.layers[i] = lz
            self.lazy.append(lz)
        self._materialize(self.embed.real, "model.language_model.embed_tokens.")
        self._materialize(self.text.norm, "model.language_model.norm.")
        self._materialize(skel.lm_head, "lm_head.")
        self.text.rotary_emb.to(self.device)          # nicht persistente Puffer (Rotation) liegen auf dem Gerät
        for i in progress(range(split, n), "Obere Schichten laden", self.verbose and self.device.type == "cuda"):
            self.text.layers[i] = self.build_layer(i)
        self._modes("pass", "inject")

    def get(self, raw):
        """Ein Gewicht unter seinem Originalnamen, auf das Gerät und in den Zieltyp gebracht."""
        if raw not in self.shard_of:
            raise KeyError(f"Gewicht nicht in den Dateien gefunden: {raw}")
        with safe_open(self.shard_of[raw], framework="pt", device=str(self.device)) as f:
            t = f.get_tensor(raw)
        return t.to(self.dtype) if t.is_floating_point() else t

    def _materialize(self, module, prefix):
        sd = {k: self.get(prefix + k) for k in module.state_dict().keys()}
        module.load_state_dict(sd, strict=True, assign=True)
        module.requires_grad_(False)

    def build_layer(self, i):
        """Schicht i mit allen Gewichten auf dem Gerät."""
        from accelerate import init_empty_weights
        from transformers.models.qwen3_5_moe.modeling_qwen3_5_moe import Qwen3_5MoeDecoderLayer
        with init_empty_weights():
            layer = Qwen3_5MoeDecoderLayer(self.tc, i)
        self._materialize(layer, f"model.language_model.layers.{i}.")
        if i < self.split:
            self.stats["layer_loads"] += 1
        return layer.to(self.device).eval()

    def _modes(self, lazy, embed):
        """Platzhalter umschalten. Läufe: ('pass', 'inject'). Vorberechnung: ('capture' oder 'stream', 'embed')."""
        for lz in self.lazy:
            lz.mode = lazy
        self.embed.mode = embed

    def injected(self, input_ids):
        """Eingabe der Schicht SPLIT für einen vorberechneten Text, für alle Zeilen des Batches."""
        if (input_ids != input_ids[0:1]).any():
            raise RuntimeError("Alle Zeilen eines Laufs müssen denselben Text enthalten.")
        key = tuple(int(x) for x in input_ids[0].tolist())
        h = self.cache.get(key)
        if h is None:
            raise RuntimeError(f"Text mit {len(key)} Token ist nicht vorberechnet: vorher STREAM.ensure([...]) "
                               "mit genau diesem Text aufrufen.")
        return h.expand(input_ids.shape[0], -1, -1).contiguous()

    @torch.no_grad()
    def ensure(self, seqs):
        """Berechnet die Eingabe der Schicht SPLIT für alle fehlenden Texte. Liefert die Anzahl neuer Texte.

        Die Texte werden nach Länge sortiert und in Blöcke zu chunk gefasst. Mit layer_major (Standard) wird jede
        untere Schicht für alle Blöcke nur einmal geladen; passen die Zwischenzustände nicht ins Budget, gibt es
        mehrere Durchläufe. Gespeichert wird erst am Ende eines Durchlaufs."""
        todo, seen = [], set()
        for s in seqs:
            key = tuple(int(x) for x in s)
            if key and key not in self.cache and key not in seen:
                seen.add(key)
                todo.append(key)
        if not todo:
            return 0
        t0, loads0 = time.time(), self.stats["layer_loads"]
        todo.sort(key=len)                              # ähnliche Längen zusammen: weniger Auffüllung
        blocks = [todo[c0:c0 + self.chunk] for c0 in range(0, len(todo), self.chunk)]
        if self.layer_major:
            per_token = self.tc.hidden_size * torch.finfo(self.dtype).bits // 8
            sweeps, cur, cur_bytes = [], [], 0
            for keys in blocks:
                need = len(keys) * len(keys[-1]) * per_token      # der längste Text steht am Blockende
                if cur and cur_bytes + need > self.state_budget:
                    sweeps.append(cur)
                    cur, cur_bytes = [], 0
                cur.append(keys)
                cur_bytes += need
            sweeps.append(cur)
            for j, sw in enumerate(sweeps):
                part = f" (Teil {j + 1} von {len(sweeps)})" if len(sweeps) > 1 else ""
                self._sweep(sw, f"Vorberechnung: {sum(len(b) for b in sw)} Texte{part}, untere Schichten")
        else:
            for keys in progress(blocks, f"Vorberechnung: {len(todo)} Texte, blockweise", self.verbose):
                self._prefill(keys)
        if self.verbose:
            print(f"Vorberechnung fertig: {len(todo)} Texte in {len(blocks)} Blöcken, "
                  f"{self.stats['layer_loads'] - loads0} Ladevorgänge unterer Schichten, {time.time() - t0:.0f} s", flush=True)
        return len(todo)

    def _padded(self, keys):
        """Token-IDs eines Blocks als [Anzahl, größte Länge], rechts aufgefüllt. Auffüllen und die anderen Zeilen
        ändern die Werte der echten Positionen nur im Rahmen der Rundung (kausale Rechnung, Zeilen unabhängig)."""
        ids = torch.full((len(keys), max(len(k) for k in keys)), self.pad_id, dtype=torch.long)
        for r, k in enumerate(keys):
            ids[r, :len(k)] = torch.tensor(k, dtype=torch.long)
        return ids.to(self.device)

    @torch.no_grad()
    def _sweep(self, blocks, desc):
        """Ein Durchlauf über mehrere Blöcke: Jede untere Schicht wird einmal geladen und rechnet alle Blöcke.
        Womit das Modell eine Schicht aufruft (Positionen, Masken), wird vorher je Block aufgezeichnet; die
        Aufrufe sind dieselben wie im Vorwärtslauf des Modells."""
        plans, layer = [], None
        try:
            for keys in blocks:
                plans.append(self._record_block(keys))
            for i in progress(range(self.split), desc, self.verbose):
                layer = self.build_layer(i)
                for p in plans:
                    p["h"] = layer(p["h"], **p["kw"][i])
                layer = None
            for p in plans:
                for r, k in enumerate(p["keys"]):
                    self.cache[k] = p["h"][r:r + 1, :len(k)].clone()
                self.stats["texts"] += len(p["keys"])
                p["h"] = None                           # Blockzustand sofort freigeben
        except BaseException as ex:
            traceback.clear_frames(ex.__traceback__)    # die Fehlermeldung soll keine geladene Schicht festhalten
            raise
        finally:                                        # auch nach einem Abbruch bleibt nichts auf dem Gerät hängen
            layer = None
            for p in plans:
                p.clear()
            plans.clear()

    @torch.no_grad()
    def _record_block(self, keys):
        """Lässt das Modell einen Block bis zur Schicht SPLIT durchlaufen, ohne die unteren Schichten zu rechnen.
        Aufgezeichnet werden die Einbettung und die Argumente, mit denen jede untere Schicht aufgerufen würde."""
        ids = self._padded(keys)
        self._rec = {"h": None, "kw": {}}

        def stop(mod, args):
            raise _StopAfterSplit()

        hook = self.text.layers[self.split].register_forward_pre_hook(stop)
        self._modes("capture", "embed")
        try:
            self.text(input_ids=ids, use_cache=False)
        except _StopAfterSplit:
            pass
        except BaseException as ex:
            traceback.clear_frames(ex.__traceback__)
            raise
        finally:
            hook.remove()
            self._modes("pass", "inject")
            rec, self._rec = self._rec, None
        if rec["h"] is None or sorted(rec["kw"]) != list(range(self.split)):
            raise RuntimeError("Vorberechnung: Die Aufzeichnung der unteren Schichten ist unvollständig.")
        return {"keys": keys, "h": rec["h"], "kw": rec["kw"]}

    def _capture(self, idx, hidden_states, kwargs):
        if idx == 0:
            self._rec["h"] = hidden_states.detach()
        self._rec["kw"][idx] = kwargs

    @torch.no_grad()
    def _prefill(self, keys):
        """Blockweise Vorberechnung (layer_major = False): Das Modell rechnet einen Block bis zur Schicht SPLIT,
        jeder Platzhalter lädt seine Schicht für diesen einen Aufruf."""
        got = {}

        def grab(mod, args):
            got["h"] = args[0].detach()
            raise _StopAfterSplit()

        ids = self._padded(keys)
        hook = self.text.layers[self.split].register_forward_pre_hook(grab)
        self._modes("stream", "embed")
        try:
            self.text(input_ids=ids, use_cache=False)
        except _StopAfterSplit:
            pass
        except BaseException as ex:
            traceback.clear_frames(ex.__traceback__)    # die Fehlermeldung soll keine geladene Schicht festhalten
            raise
        finally:
            hook.remove()
            self._modes("pass", "inject")
        H = got["h"]
        for r, k in enumerate(keys):
            self.cache[k] = H[r:r + 1, :len(k)].clone()
        self.stats["texts"] += len(keys)
        del got, H

    def gpu_gib(self):
        return torch.cuda.memory_allocated(self.device) / 2**30 if self.device.type == "cuda" else 0.0


def check_session(rc, streamer):
    """Bricht mit einer klaren Meldung ab, wenn Routing-Steuerung oder Modell noch aus einer früheren Fassung oder
    einer früheren Ausführung der Bibliothekszelle stammen."""
    if not isinstance(rc, RoutingControl) or not isinstance(streamer, LayerStreamer):
        raise RuntimeError("Bitte zuerst die Modellzelle erneut ausführen. Sie lädt das Modell nicht neu, "
                           "sondern übernimmt es für die jetzige Fassung der Bibliothek.")


def adopt_streamer(st):
    """Bringt einen bereits geladenen LayerStreamer auf den Stand der jetzt definierten Klassen, zum Beispiel
    nachdem die Bibliothekszelle erneut ausgeführt wurde. Gewichte und gespeicherte Eingaben bleiben; es wird
    nichts neu geladen."""
    st.__class__ = LayerStreamer
    for lz in st.lazy:
        lz.__class__ = LazyLayer
    st.embed.__class__ = StreamEmbed
    for name, default in (("layer_major", True), ("state_budget", 2 * 2**30), ("verbose", True), ("_rec", None)):
        if not hasattr(st, name):
            setattr(st, name, default)
    if not hasattr(st, "stats"):
        st.stats = {"layer_loads": 0, "texts": 0}
    st._modes("pass", "inject")
    return st

In [ ]:
# ---------- Modell laden: schichtweise, passt auf eine GPU mit 40 GB ----------
from transformers import AutoTokenizer
if TEST_MODE:
    assert "STREAM" in globals() and "tok" in globals(), "Testmodus: STREAM und tok müssen bereitgestellt werden"
    STREAM = adopt_streamer(STREAM)
    model = STREAM.model
else:
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    LOADED = globals().get("STREAM")
    STREAM, model = LOADED, None                 # ein geladenes Modell bleibt in jedem Fall erhalten
    if P1_FROM_FILES:
        DEV = torch.device("cpu")
        E_NUM, I_NUM = 256, 512
        print("Nur Teil 1 (ohne Modell). Teile 0, 2 und 3 brauchen P1_FROM_FILES = False.")
    elif LOADED is not None:
        # Die Zelle wurde erneut ausgeführt: Das geladene Modell bleibt. Ein zweites Laden würde den GPU-Speicher sprengen.
        STREAM = LOADED
        if STREAM.split != SPLIT_LAYER:
            raise RuntimeError("Ein Modell mit anderer Aufteilung ist geladen. Für eine neue Aufteilung die Laufzeit neu starten.")
        if getattr(STREAM, "model_id", MODEL_ID) != MODEL_ID:
            raise RuntimeError(f"Geladen ist {STREAM.model_id}, eingestellt {MODEL_ID}. Für ein anderes Modell die Laufzeit neu starten.")
        STREAM = adopt_streamer(STREAM)
        model = STREAM.model
        print("Das Modell ist bereits geladen und wird weiterverwendet (zum Neuladen die Laufzeit neu starten).")
    else:
        import gc, os, shutil
        from huggingface_hub import hf_hub_download, snapshot_download
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        from transformers import AutoConfig
        assert torch.cuda.is_available(), "Keine GPU gefunden: in Colab unter Laufzeit eine GPU (A100) wählen."
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {torch.cuda.get_device_name(0)}, {gb:.1f} GB")
        if gb < MIN_GPU_GB:
            raise RuntimeError(f"Zu wenig GPU-Speicher: {gb:.1f} GB, nötig sind mindestens {MIN_GPU_GB} GB.")
        cfg_full = AutoConfig.from_pretrained(MODEL_ID)
        tc = cfg_full.text_config
        expect = dict(num_hidden_layers=40, hidden_size=2048, num_experts=256, num_experts_per_tok=8, moe_intermediate_size=512)
        for k, v in expect.items():
            assert getattr(tc, k) == v, f"{k} = {getattr(tc, k)}, erwartet {v}"
        skel = skeleton(cfg_full)                # Aufbau ohne Gewichte
        names = needed_names(skel)
        wm = json.load(open(hf_hub_download(MODEL_ID, "model.safetensors.index.json"), encoding="utf-8"))["weight_map"]
        missing = [n for n in names if n not in wm]
        assert not missing, f"Gewichte fehlen im Checkpoint: {missing[:5]}"
        files = sorted({wm[n] for n in names})
        free_gb = shutil.disk_usage("/").free / 1e9
        print(f"Festplatte frei: {free_gb:.0f} GB (der Download braucht rund 72 GB)")
        if free_gb < 74:
            raise RuntimeError("Zu wenig Festplattenplatz für die Gewichte (rund 72 GB nötig).")
        local = snapshot_download(MODEL_ID, allow_patterns=files)
        shard_of = {n: os.path.join(local, wm[n]) for n in names}
        STREAM = LayerStreamer(skel, shard_of, "cuda", torch.bfloat16, SPLIT_LAYER, STREAM_CHUNK)
        STREAM.model_id = MODEL_ID
        model = STREAM.model
        print(f"Obere Schichten ab {SPLIT_LAYER} liegen auf der GPU: {STREAM.gpu_gib():.1f} GiB belegt")
if model is not None:
    assert STREAM.split <= min(c[0] for c in CANDIDATES), "Kandidaten müssen in den oberen, dauerhaft geladenen Schichten liegen"
    DEV = STREAM.device
    tc = model.config.text_config
    E_NUM, I_NUM = tc.num_experts, tc.moe_intermediate_size
    remove_neuron_hooks(model)               # nichts aus einem früheren, abgebrochenen Lauf bleibt am Modell hängen
    remove_routing_hooks(model)              # auch nicht aus einer früheren Fassung; keine doppelten Routing-Hooks
    RC = RoutingControl(model)               # steuert das Routing aller oberen Schichten
V_TOK = len(tok)                             # nur echte Token (das Modell hat Auffüll-Zeilen darüber)
print("Bereit | Modell:", "geladen" if model is not None else "nicht geladen", "| Gerät", DEV,
      "| Experten", E_NUM, "| Neuronen je Experte", I_NUM, "| Token", V_TOK)

In [ ]:
# ---------- Zielwörter als einzelne Token-IDs ----------
def family_ids(words):
    out = []
    for w in words:
        ids_w = tok.encode(w, add_special_tokens=False)
        assert len(ids_w) == 1, f"{w!r} ist kein einzelnes Token: {ids_w}"
        out.append(ids_w[0])
    return out

FAM_KL_IDS = family_ids(FAM_KL)
FAM_CTL_IDS = family_ids(FAM_CTL)
print("KL-Zielwörter:", [tok.decode([i]) for i in FAM_KL_IDS])
print("Kontroll-Zielwörter:", [tok.decode([i]) for i in FAM_CTL_IDS])

## Teil 0: Kalibrierung gegen den Atlas

Der Atlas speichert für jede Feuerung ein Token-Fenster. Die Werte im Fenster sind die Aktivierung **in Vielfachen von θ**, auf zwei Nachkommastellen gerundet. Positionen vor `u` sind nur Kontext ohne Wert; `null` bedeutet, dass der Router den Experten dort nicht gewählt hat. Der Feuer-Token (Position `f`) hat einen Wert, der nicht die Feuerung selbst ist, und wird deshalb ausgelassen.

**Ablauf.** Alle Fenster laufen zuerst gemeinsam durch die unteren Schichten: Jede der 21 Schichten wird einmal geladen, eine Fortschrittsanzeige zählt mit. Danach läuft jedes Fenster einzeln durch die oberen Schichten. Unter dem Ergebnis steht eine beschreibende Diagnose (Zusammenhang, Übereinstimmung nach Fensterart und Position, andere denkbare Definitionen des gespeicherten Werts). Sie ändert die Regel nicht.

**Fensterart.** Der Atlas zeigt je Feuerung die 100 Token davor. 827 der 950 Fenster sind aus einem längeren Text geschnitten (`u` = 84); der Text davor fehlt im Fenster, der Atlas hat ihn aber mitgerechnet (wortgleiche Fenster tragen verschiedene Werte). Fenster mit `u` < 84 beginnen am Textanfang und haben den vollen Kontext. Die Diagnose weist beide Arten getrennt aus.

**Gate 0b.** Unter Gate 0 steht Gate 0b: dieselben Werte, aber nur die Fenster ab Textanfang, geprüft auf Zusammenhang (r ≥ 0,99), Steigung (0,95 bis 1,05) und Routing (≥ 98 %). Es ist nachträglich festgelegt und im Bericht als Abweichung vom Plan anzugeben. Mit `GATE0B = False` gilt wieder allein Gate 0.

**Hinweis zur Ausgabe.** Beim ersten Lauf meldet transformers, dass `causal_conv1d` nicht installiert ist und seine Referenz-Implementierung benutzt wird. Das ist kein Fehler. Die Referenz ist ein einzelner Faltungsaufruf je Schicht; mathematisch ist es dieselbe Rechnung, nur die Rundung kann minimal abweichen.

In [ ]:
# ---------- Gate 0: Kalibrierung gegen die gespeicherten Atlas-Aktivierungen ----------
from neuron_atlas import notebook
from neuron_atlas.upstream import pole_path

CALIB_OK, GATE0_STATS, CALIB_B_OK, GATE0B_STATS = None, None, None, None
if TEST_MODE or model is None:
    print("Kalibrierung übersprungen (Testmodus oder kein Modell geladen)")
else:
    atlas = notebook.open_atlas()
    rel = pole_path("landscape", CALIB_POLE)
    srel = pole_path("subcircuit", CALIB_POLE)
    fetched = atlas.source.fetch_many([rel, srel])
    bad = [r.relpath for r in fetched if r.status not in ("present", "absent")]
    assert not bad, f"Atlas-Dateien nicht lesbar: {bad}"
    dat = json.loads(atlas.source.store.read(rel))
    THETA = float(json.loads(atlas.source.store.read(srel))["theta"])
    print(f"theta = {THETA} (aus {srel})")
    dots = dat["dots"][:25] if SMOKE else dat["dots"]
    windows = []                                   # (Punkt, Token-IDs, exakt übersetzt?)
    for dot in dots:
        t = dot["t"]
        ids_list = tok.encode("".join(s for s, _ in t), add_special_tokens=False)
        exact = len(ids_list) == len(t) and [tok.decode([i]) for i in ids_list] == [s for s, _ in t]
        windows.append((dot, ids_list, exact))
    check_session(RC, STREAM)
    remove_neuron_hooks(model)
    RC.reset()
    STREAM.ensure([ids for _, ids, exact in windows if exact])   # alle Fenster in einem Durchlauf der unteren Schichten
    hookC = NeuronHook(model, 24, 124, 345, RC)
    rows, skipped = [], 0
    shifted = {-1: [], 1: []}                      # nur für die Diagnose: Wert der Nachbarposition gegen den Atlas-Wert
    try:
        for dot, ids_list, exact in progress(windows, "Gate 0: Fenster rechnen"):
            t, u, f = dot["t"], dot["u"], dot["f"]
            if not exact:
                skipped += 1                       # Fenster lässt sich nicht exakt zurück in Token übersetzen
                continue
            act_m, w_m = activation_at(model, hookC, RC, torch.tensor(ids_list, device=DEV))
            for j in range(u, len(t)):
                if j == f:
                    continue
                rows.append(dict(a=float(act_m[j]) / THETA, w=float(w_m[j]), routed=float(w_m[j]) > 0, v=t[j][1],
                                 pos=j - u, left=j))
                if t[j][1] is not None:
                    for d in (-1, 1):
                        if 0 <= j + d < len(t):
                            shifted[d].append((float(act_m[j + d]) / THETA, t[j][1]))
    finally:
        hookC.remove()
    routed = [r for r in rows if r["v"] is not None]
    ok_val = [abs(r["a"] - r["v"]) <= CAL_ABS + CAL_REL * abs(r["v"]) for r in routed]
    frac_value = sum(ok_val) / len(routed)
    agree = sum(r["routed"] == (r["v"] is not None) for r in rows) / len(rows)
    print(f"Fenster übernommen: {len(dots) - skipped} von {len(dots)} (nicht exakt übersetzt: {skipped})")
    print(f"Positionen mit Atlas-Wert (ohne Kontext, ohne Feuer-Token): {len(rows)}, davon gerouted: {len(routed)}")
    print(f"Wert innerhalb 0,005 + 5 %: {frac_value:.3f}   (Gate: >= {GATE_VALUE})")
    print(f"Routing stimmt überein:     {agree:.3f}   (Gate: >= {GATE_ROUTE})")
    # Beschreibende Diagnose. Sie ändert die Regel nicht; sie zeigt, woran es liegt, falls das Gate nicht besteht.
    def share_ok(pairs):
        return sum(abs(a - v) <= CAL_ABS + CAL_REL * abs(v) for a, v in pairs) / max(len(pairs), 1)

    def fin(x, nd=4):
        return round(float(x), nd) if math.isfinite(float(x)) else None

    GATE0_STATS = dict(windows=len(dots) - skipped, slots=len(rows), slots_routed_in_atlas=len(routed),
                       share_value_ok=round(frac_value, 4), share_routing_ok=round(agree, 4))
    if len(routed) > 2:
        A_ = torch.tensor([r["a"] for r in routed], dtype=torch.float64)
        V_ = torch.tensor([r["v"] for r in routed], dtype=torch.float64)
        err = (A_ - V_).abs()
        GATE0_STATS.update(pearson_r=fin(torch.corrcoef(torch.stack([A_, V_]))[0, 1]),
                           slope_through_origin=fin((A_ * V_).sum() / (V_ * V_).sum().clamp_min(1e-30)),
                           median_abs_error=fin(err.median()),
                           q90_abs_error=fin(torch.quantile(err, 0.9)))
        print(f"Diagnose (beschreibend): Pearson r = {GATE0_STATS['pearson_r']}, Steigung a gegen v durch 0 = "
              f"{GATE0_STATS['slope_through_origin']}, Median |a - v| = {GATE0_STATS['median_abs_error']}, "
              f"90-%-Quantil = {GATE0_STATS['q90_abs_error']}")
    print("  Gate-Kriterien nach Fensterart:")
    by_type = {}
    for name, sel in (("Fenster ab Textanfang (u < 84, voller Kontext)", lambda r: r["left"] - r["pos"] < 84),
                      ("abgeschnittene Fenster (u = 84, Text davor fehlt)", lambda r: r["left"] - r["pos"] >= 84)):
        allp = [r for r in rows if sel(r)]
        part = [(r["a"], r["v"]) for r in allp if r["v"] is not None]
        if allp:
            ag = sum(r["routed"] == (r["v"] is not None) for r in allp) / len(allp)
            by_type[name] = dict(share_value_ok=fin(share_ok(part)), n_valued=len(part), share_routing_ok=fin(ag), n_slots=len(allp))
            print(f"    {name}: Wert {share_ok(part):.3f} innerhalb (n={len(part)}), Routing gleich {ag:.3f} (n={len(allp)})")
    print("  Übereinstimmung nach Abstand zum ersten Wert im Fenster (Position minus u):")
    by_pos = {}
    for lo_, hi_ in ((0, 8), (8, 30), (30, 10**9)):
        part = [ok for ok, r in zip(ok_val, routed) if lo_ <= r["pos"] < hi_]
        if part:
            label = f"{lo_} bis {hi_ - 1 if hi_ < 10**9 else '…'}"
            by_pos[label] = [round(sum(part) / len(part), 4), len(part)]
            print(f"    {label}: {by_pos[label][0]:.3f} (n={len(part)})")
    print("  Übereinstimmung nach Kontext links im Fenster (Zahl der Token vor der Position):")
    by_left = {}
    for lo_, hi_ in ((0, 8), (8, 16), (16, 32), (32, 64), (64, 10**9)):
        part = [ok for ok, r in zip(ok_val, routed) if lo_ <= r["left"] < hi_]
        if part:
            label = f"{lo_} bis {hi_ - 1 if hi_ < 10**9 else '…'}"
            by_left[label] = [round(sum(part) / len(part), 4), len(part)]
            print(f"    {label}: {by_left[label][0]:.3f} (n={len(part)})")
    alt = {"Aktivierung mal Routing-Gewicht": share_ok([(r["a"] * r["w"], r["v"]) for r in routed]),
           "Aktivierung ohne Teilen durch theta": share_ok([(r["a"] * THETA, r["v"]) for r in routed]),
           "Position um eins früher": share_ok(shifted[-1]),
           "Position um eins später": share_ok(shifted[1])}
    for name, val in alt.items():
        print(f"  Wäre der Atlas-Wert »{name}«: {val:.3f} innerhalb der Toleranz")
    n_both = sum(r["routed"] and r["v"] is not None for r in rows)
    n_atlas = sum((not r["routed"]) and r["v"] is not None for r in rows)
    n_model = sum(r["routed"] and r["v"] is None for r in rows)
    print(f"  Routing: in beiden gewählt {n_both}, nur im Atlas {n_atlas}, nur im Modell {n_model}, "
          f"in keinem {len(rows) - n_both - n_atlas - n_model}")
    GATE0_STATS.update(by_window_type=by_type, share_ok_by_distance_from_u=by_pos,
                       share_ok_by_left_context=by_left, share_ok_under_other_definitions={k: round(v, 4) for k, v in alt.items()},
                       routing_both=n_both, routing_only_atlas=n_atlas, routing_only_model=n_model)
    CALIB_OK = frac_value >= GATE_VALUE and agree >= GATE_ROUTE
    print("GATE 0 BESTANDEN" if CALIB_OK else "GATE 0 NICHT BESTANDEN", "(vorab festgelegte Regel)")
    # Gate 0b: nachträglich festgelegt (28.11.2026), im Bericht als Abweichung vom Plan. Nur Fenster ab Textanfang.
    use_b = globals().get("GATE0B", True)
    b_r, b_slope, b_route = (globals().get("GATE0B_R", 0.99), globals().get("GATE0B_SLOPE", (0.95, 1.05)),
                             globals().get("GATE0B_ROUTE", 0.98))
    GATE0B_STATS = calibration_fit(rows)
    CALIB_B_OK = calibration_b_ok(GATE0B_STATS, b_r, b_slope, b_route)
    show = lambda x: f"{x:.4f}" if x is not None else "nicht berechenbar"
    print(f"Gate 0b (nachträglich, nur Fenster ab Textanfang: {GATE0B_STATS['n_valued']} Werte, {GATE0B_STATS['n_slots']} Positionen): "
          f"r = {show(GATE0B_STATS['pearson_r'])} (>= {b_r}), Steigung = {show(GATE0B_STATS['slope_through_origin'])} "
          f"({b_slope[0]} bis {b_slope[1]}), Routing gleich = {show(GATE0B_STATS['share_routing_ok'])} (>= {b_route})")
    print("GATE 0b BESTANDEN" if CALIB_B_OK else "GATE 0b NICHT BESTANDEN", "(nachträglich festgelegt)",
          "| entscheidet über den Abbruch" if use_b else "| nur berichtet (GATE0B = False)")
    decisive = CALIB_B_OK if use_b else CALIB_OK
    if not decisive and STOP_IF_CALIBRATION_FAILS:
        raise RuntimeError(("Gate 0b" if use_b else "Gate 0") + " nicht bestanden: Hook oder Definition stimmen nicht mit dem Atlas überein.")
    if use_b and not CALIB_OK:
        print("Weiter nach Gate 0b. Im Bericht: Gate 0 nach der vorab festgelegten Regel nicht bestanden; Abweichung vom Plan.")

## Teil 1: Direkte Ausgabewirkung auf KL-Wörter (ohne Vorwärtslauf)

Für jedes Neuron rechnen wir `W_U · (Endnorm-Gewicht ⊙ Schreibvektor)`. Das ist die Wirkung auf alle echten Vokabel-Token. Der Atlas nennt diese Liste `static`; wir prüfen, ob sie genau diese Wirkung ist (in früheren Läufen für 57 Neuronen mit r = 1,000000 bestätigt). Die Kennzahl ist der Mittelwert der z-Werte der KL-Wörter. Verglichen wird mit allen Neuronen der Schicht und mit zufälligen Dreiergruppen aus dem Vokabular.

In [ ]:
# ---------- Teil 1: Gewichte (aus dem Modell oder per Bereichsanfrage) ----------
def weights_for_p1():
    layers = sorted({c[0] for c in CANDIDATES})
    if not P1_FROM_FILES:
        W_U = model.lm_head.weight.detach().float().cpu()[:V_TOK]
        gain = (1.0 + text_model(model).norm.weight.detach().float()).cpu()
        Wd = {L: moe_block(model, L).experts.down_proj.detach().float().cpu() for L in layers}
        return W_U, gain, Wd
    import urllib.request
    BASE = f"https://huggingface.co/{MODEL_ID}/resolve/main/"
    with urllib.request.urlopen(BASE + "model.safetensors.index.json") as r:
        IDX = json.loads(r.read())["weight_map"]

    def get(name):
        return read_safetensors_tensors(BASE + IDX[name], [name])[name]

    gain = 1.0 + get("model.language_model.norm.weight").float()
    W_U = get("lm_head.weight").float()[:V_TOK]
    Wd = {L: get(f"model.language_model.layers.{L}.mlp.experts.down_proj").float() for L in layers}
    return W_U, gain, Wd

W_U, gain, Wd_by_layer = weights_for_p1()
P1 = {"candidates": {}, "null": {}, "static_check": None}
for L in sorted(Wd_by_layer):
    Wd = Wd_by_layer[L]
    zK, _, _ = direct_effect_columns(W_U, gain, Wd, FAM_KL_IDS)
    zC, _, _ = direct_effect_columns(W_U, gain, Wd, FAM_CTL_IDS)
    for (LL, e, n) in CANDIDATES:
        if LL != L:
            continue
        idx = e * Wd.shape[2] + n
        v = direct_effect_vector(W_U, gain, Wd[e][:, n])
        z = (v - v.mean()) / v.std(unbiased=False)
        P1["candidates"][f"L{L}_E{e}_N{n}"] = {
            "KL_family_z_mean": round(float(zK[idx]), 3),
            "KL_percentile_in_layer": round(percentile_rank(zK, zK[idx]), 5),
            "control_family_z_mean": round(float(zC[idx]), 3),
            "control_percentile_in_layer": round(percentile_rank(zC, zC[idx]), 5),
            "KL_token_z": {tok.decode([i]): round(float(z[i]), 3) for i in FAM_KL_IDS},
            "top_promoted": [[tok.decode([i]), round(float(z[i]), 2)] for i in torch.topk(z, 10).indices.tolist()],
            "top_suppressed": [[tok.decode([i]), round(float(z[i]), 2)] for i in torch.topk(-z, 10).indices.tolist()],
        }
    rng = random.Random(SEED)
    fams = torch.tensor([rng.sample(range(1000, V_TOK), 3) for _ in range(300)] + [FAM_KL_IDS])
    best, arg = family_max_null(W_U, gain, Wd, fams)
    kl_max = float(best[-1])
    rnd = best[:-1]
    q99 = float(torch.quantile(rnd, 0.99))
    norms = W_U.norm(dim=1)
    pct = {tok.decode([i]): round(percentile_rank(norms, norms[i]), 4) for i in FAM_KL_IDS}
    P1["null"][f"L{L}"] = {
        "KL_family_max_neuron_z": round(kl_max, 3),
        "random_triplets_max_neuron_z_quantiles": {str(q): round(float(torch.quantile(rnd, q)), 3) for q in (0.5, 0.9, 0.95, 0.99)},
        "KL_rank_among_301_families(1=highest)": int((best >= kl_max).sum()),
        "KL_argmax_neuron": [int(arg[-1]) // Wd.shape[2], int(arg[-1]) % Wd.shape[2]],
        "P1a_auffaellig": bool(kl_max > q99),
        "KL_token_norm_percentile": pct,
        "P1b_auffaellig": bool(any(p >= 0.99 or p <= 0.01 for p in pct.values())),
    }

if not TEST_MODE and 24 in Wd_by_layer:
    if "atlas" not in globals():
        atlas = notebook.open_atlas()
    from neuron_atlas.upstream import neuron_path
    mrel = neuron_path("circuit_meta", "L24_E124_N345")
    fm = atlas.source.fetch_many([mrel])
    assert all(r.status in ("present", "absent") for r in fm), "Schaltkreis-Datei nicht lesbar"
    meta = json.loads(atlas.source.store.read(mrel))
    static = [(r["token_id"], r["w"]) for r in meta["logits"] if r["source"] in ("static", "both")]
    vv = direct_effect_vector(W_U, gain, Wd_by_layer[24][124][:, 345])
    tids = torch.tensor([t for t, _ in static])
    ws = torch.tensor([w for _, w in static], dtype=torch.float32)
    P1["static_check"] = {
        "rows": len(static),
        "sign_agreement": round(float(((ws > 0) == (vv[tids] > 0)).float().mean()), 4),
        "pearson_r": round(float(torch.corrcoef(torch.stack([ws, vv[tids]]))[0, 1]), 6),
        "max_abs_diff": round(float((ws - vv[tids]).abs().max()), 6),
    }
print(json.dumps(P1, indent=1, ensure_ascii=False))

## Teil 2: Kausal. Ausschalten an der Zielposition

**Satzpaare.** Jeder KL-Satz hat einen Kontrollsatz mit demselben Aufbau, in dem der Fachbegriff ausgetauscht ist (`Kullback-Leibler divergence` gegen `Euclidean distance`, `Hamming distance`, `mean squared error` oder `Jensen-Shannon distance`). Der Ersatz wird pro Satz so gewählt, dass die Tokenzahl möglichst gleich ist. Paare ohne Zielposition in einem der beiden Sätze werden ausgelassen und gezählt.

**Messung.** Das Neuron wird an der Position ausgeschaltet, die das Zielwort vorhersagt. Gemessen wird die Änderung der Log-Wahrscheinlichkeit des Zielworts. Das Routing aller Schichten ist festgehalten (primär). Der freie Lauf (wie der Atlas) steht daneben.

**Kontrollen.** Pro Kandidat werden 19 Neuronen desselben Experten gewählt, die ihm in der mittleren Aktivität am nächsten liegen. Sie teilen das Routing, unterscheiden sich aber in Aktivität und Ausgaberichtung.

**Abbruch und Fortsetzen.** Die Läufe-Zelle schreibt nach jedem fertigen Neuron den Zwischenstand in `kl_unembedding_partial.json`. Wird sie nach einem Abbruch erneut ausgeführt, rechnet sie nur das, was noch fehlt, solange Kandidaten, Satzpaare, Kontrollen und Einstellungen dieselben sind. Die Modellzelle lädt nicht neu, wenn das Modell schon geladen ist.

In [ ]:
# ---------- Satzpaare (von Hand geschrieben, keine fremden Texte) ----------
KL_SENTENCES = [
    "To measure how far the predicted distribution is from the true one, we compute the Kullback-Leibler divergence.",
    "The variational lower bound contains a penalty term equal to the Kullback-Leibler divergence between the posterior and the prior.",
    "Minimising the forward Kullback-Leibler divergence forces the model to cover every mode of the data.",
    "For two univariate Gaussians the closed form of the Kullback-Leibler divergence follows from the log-variance ratio.",
    "The distillation loss is the Kullback-Leibler divergence between the teacher and the student output distributions.",
    "Each sparse autoencoder layer adds a penalty, implemented as a Kullback-Leibler divergence between the target and the observed activation rates.",
    "A large Kullback-Leibler divergence indicates that the two language model distributions disagree strongly on the next token.",
    "The policy update is constrained by a bound on the Kullback-Leibler divergence between the old and the new policy.",
    "Because the Kullback-Leibler divergence is asymmetric, reversing the two arguments changes the value of the loss.",
    "We report the average Kullback-Leibler divergence over all test sequences, computed from the token probabilities.",
    "Information gain is the expected Kullback-Leibler divergence between the posterior and the prior beliefs.",
    "The evidence lower bound decomposes into a reconstruction term and a Kullback-Leibler divergence term.",
    "Temperature scaling changes the Kullback-Leibler divergence between the softened teacher and the student distributions.",
    "In the experiments the Kullback-Leibler divergence between the clean and the ablated predictions was summed over sixteen positions.",
    "The gradient of the Kullback-Leibler divergence with respect to the logits equals the difference of the two probability vectors.",
    "Entropy regularisation adds a bonus term, while the Kullback-Leibler divergence to a reference policy acts as a penalty.",
    "The score measures the Kullback-Leibler divergence between the empirical and the model distribution of the tokens.",
    "Calibration error can be estimated from the Kullback-Leibler divergence between predicted and observed frequencies.",
    "Variational inference fits a simpler family by minimising the Kullback-Leibler divergence between the approximate and the true posterior.",
    "The Kullback-Leibler divergence is zero exactly when the two distributions coincide on every outcome.",
    "Stochastic estimates of the Kullback-Leibler divergence are unbiased when the samples come from the reference distribution.",
    "Our regulariser penalises the Kullback-Leibler divergence between the attention distributions of the two heads.",
    "The reward model is calibrated by minimising the Kullback-Leibler divergence to the human preference distribution.",
    "We approximate the intractable posterior with a Gaussian and report its Kullback-Leibler divergence to the prior.",
    "Mutual information can be written as an expected Kullback-Leibler divergence between a joint and a product distribution.",
    "The diffusion loss reduces to a sum of Kullback-Leibler divergences between consecutive noise levels.",
    "Model merging is evaluated by the Kullback-Leibler divergence between the merged and the individual expert predictions.",
    "Temperature-matched softmax outputs give a smaller Kullback-Leibler divergence than the raw logits do.",
    "A hidden Markov model is trained by minimising a free energy that contains the Kullback-Leibler divergence to the approximate posterior.",
    "The Kullback-Leibler divergence penalises confident predictions that put little mass on the true outcome.",
]
REPLACEMENTS = ["Euclidean distance", "Hamming distance", "mean squared error", "Jensen-Shannon distance"]

def make_control(s, repl):
    out = s.replace("Kullback-Leibler divergence", repl).replace("KL divergence", repl)
    assert "Kullback" not in out and "KL" not in out and "divergence" not in out, f"Rest im Kontrollsatz: {out}"
    return out

pairs_raw = []
for s in KL_SENTENCES:
    kl_ids = tok.encode(s, add_special_tokens=False)
    best = None
    for repl in REPLACEMENTS:
        c = make_control(s, repl)
        c_ids = tok.encode(c, add_special_tokens=False)
        diff = abs(len(c_ids) - len(kl_ids))
        if best is None or diff < best[0]:
            best = (diff, c, c_ids)
    pairs_raw.append((s, kl_ids, best[1], best[2]))

PAIRS = []
dropped = 0
for s, kl_ids, ctl, ctl_ids in pairs_raw:
    kl_pos = term_positions(kl_ids, FAM_KL_IDS)
    ctl_pos = term_positions(ctl_ids, FAM_CTL_IDS)
    if kl_pos and ctl_pos:
        PAIRS.append((kl_ids, kl_pos, ctl_ids, ctl_pos))
    else:
        dropped += 1
if SMOKE:
    PAIRS = PAIRS[:3]
len_diff = [abs(len(a) - len(c)) for a, _, c, _ in PAIRS]
print(f"Satzpaare: {len(PAIRS)} (ausgelassen wegen fehlender Zielposition: {dropped})")
print(f"Tokenlängen KL / Kontrolle: Mittel {sum(len(a) for a, *_ in PAIRS) / len(PAIRS):.1f} / "
      f"{sum(len(c) for _, _, c, _ in PAIRS) / len(PAIRS):.1f}; Längenunterschied Mittel {sum(len_diff) / len(len_diff):.2f}, Maximum {max(len_diff)}")
print("Beispiel:")
print("  KL:      ", tok.decode(PAIRS[0][0]))
print("  Kontrolle:", tok.decode(PAIRS[0][2]))

In [ ]:
# ---------- Aktivitätsgleiche Kontroll-Neuronen (gleicher Experte, gleiches Routing) ----------
def matched_neurons(L, e, n_cand, texts_ids, k):
    """Die k Neuronen desselben Experten, deren mittlere Aktivität (an gerouteten Positionen) dem Kandidaten am nächsten liegt."""
    STREAM.ensure(texts_ids)
    bi = BlockInput(model, L)
    RC.mode = "free"
    acc = torch.zeros(I_NUM, dtype=torch.float64)
    cnt = 0
    mlp_L = moe_block(model, L)
    try:
        with torch.no_grad():
            for ids_list in texts_ids:
                ids = torch.tensor(ids_list, device=DEV)
                model(input_ids=ids[None], use_cache=False)
                x = bi.x
                A = expert_acts_all(mlp_L, e, x).abs()
                sc, ix = RC.last[L]
                m = ((sc * (ix == e)).sum(-1) > 0)
                acc += A[m].sum(0).double().cpu()
                cnt += int(m.sum())
    finally:
        bi.remove()
    mean_act = acc / max(cnt, 1)
    cand_val = float(mean_act[n_cand])
    diffs = sorted((abs(math.log(float(mean_act[j]) + 1e-9) - math.log(cand_val + 1e-9)), j)
                   for j in range(I_NUM) if j != n_cand)
    return [j for _, j in diffs[:k]], cand_val

CTRL = {}
if model is not None:
    check_session(RC, STREAM)
    remove_neuron_hooks(model)
    RC.reset()
    all_ids = [p[0] for p in PAIRS] + [p[2] for p in PAIRS]
    for (L, e, n) in CANDIDATES:
        ms, cv = matched_neurons(L, e, n, all_ids, N_MATCH)
        CTRL[(L, e, n)] = [(e, j) for j in ms]
        print(f"L{L}_E{e}_N{n}: mittlere Aktivität {cv:.4f}; Kontrollen (Experte {e}): {ms}")

In [ ]:
# ---------- Läufe: Kandidaten (zwei Arme) und Kontrollen (festgehaltenes Routing) ----------
assert model is not None, "Dieser Teil braucht das Modell (P1_FROM_FILES = False)"
import hashlib
check_session(RC, STREAM)
remove_neuron_hooks(model)
RC.reset()
CTRL_RUNS = None                             # erst am Ende der Zelle gesetzt; vorher kann die Statistik nicht laufen
STREAM.ensure([p[0] for p in PAIRS] + [p[2] for p in PAIRS])

def run_text(hook, ids_list, pos, freeze):
    ids = torch.tensor(ids_list, device=DEV)
    tgt = torch.tensor(ids_list[1:] + [-1])
    tab = ablation_table(model, hook, RC, ids, tgt, W=W_WINDOW, batch=BATCH, freeze=freeze)
    return [float(tab["lp_tgt"][p, p] - tab["clean_tgt"][p]) for p in pos]

def run_neuron(L, e, n, arm):
    hook = NeuronHook(model, L, e, n, RC)
    freeze = "clean" if arm == "frozen" else None
    per_pair = []
    try:
        for kl_ids, kl_pos, ctl_ids, ctl_pos in PAIRS:
            d_kl = run_text(hook, kl_ids, kl_pos, freeze)
            d_ctl = run_text(hook, ctl_ids, ctl_pos, freeze)
            per_pair.append(sum(d_kl) / len(d_kl) - sum(d_ctl) / len(d_ctl))   # gepaarter Kontrast pro Satzpaar
    finally:
        hook.remove()
    return per_pair

# Fertige Läufe bleiben erhalten, solange Kandidaten, Satzpaare, Kontrollen und Einstellungen dieselben sind.
# Nach einem Abbruch setzt ein erneutes Ausführen dieser Zelle dort fort, wo sie stand. Der Zwischenstand steht
# zusätzlich in kl_unembedding_partial.json (zum Herunterladen, falls die Sitzung endet).
RUN_SIG = hashlib.sha256(repr((LIB_VERSION, CANDIDATES, PAIRS, sorted(CTRL.items()), W_WINDOW, BATCH)).encode()).hexdigest()
_prev = globals().get("RUN_STATE")
if _prev is None or _prev.get("sig") != RUN_SIG:
    if _prev is not None and (_prev["runs"] or _prev["ctrl"]):
        RUN_STATE_PREV = _prev
        print(f"Hinweis: {len(_prev['runs']) + len(_prev['ctrl'])} fertige Läufe passen nicht zu den jetzigen Einstellungen "
              "und werden nicht verwendet (sie bleiben in RUN_STATE_PREV).")
    RUN_STATE = {"sig": RUN_SIG, "runs": {}, "ctrl": {}}
RUNS, CTRL_DONE = RUN_STATE["runs"], RUN_STATE["ctrl"]

def save_partial():
    part = {"sig": RUN_SIG, "n_pairs": len(PAIRS),
            "runs": {f"L{k[0]}_E{k[1]}_N{k[2]}_{k[3]}": v for k, v in RUNS.items()},
            "controls": {f"L{k[0]}_E{k[1]}_N{k[2]}__E{k[3]}_N{k[4]}": v for k, v in CTRL_DONE.items()}}
    with open("kl_unembedding_partial.json", "w", encoding="utf-8") as fh:
        json.dump(part, fh)

t0 = time.time()
for (L, e, n) in CANDIDATES:
    for arm in ("frozen", "free"):
        if (L, e, n, arm) not in RUNS:
            RUNS[(L, e, n, arm)] = run_neuron(L, e, n, arm)
            save_partial()
        print(f"Kandidat L{L}_E{e}_N{n} [{arm}] fertig nach {time.time() - t0:.0f} s", flush=True)
    for ce, cj in progress(CTRL[(L, e, n)], f"Kontrollen L{L}_E{e}_N{n}"):
        if (L, e, n, ce, cj) not in CTRL_DONE:
            CTRL_DONE[(L, e, n, ce, cj)] = run_neuron(L, ce, cj, "frozen")
            save_partial()
    print(f"Kontrollen für L{L}_E{e}_N{n}: {len(CTRL[(L, e, n)])} fertig nach {time.time() - t0:.0f} s", flush=True)
CTRL_RUNS = {c: [CTRL_DONE[(*c, ce, cj)] for ce, cj in CTRL[c]] for c in CANDIDATES}

In [ ]:
# ---------- Statistik und Entscheidung nach den vorab festgelegten Regeln ----------
def summarize(per_pair):
    est, lo, hi = paired_bootstrap([[d] for d in per_pair], n_boot=5000, seed=SEED)
    sd = (sum((d - est) ** 2 for d in per_pair) / (len(per_pair) - 1)) ** 0.5
    return est, lo, hi, sd

assert globals().get("CTRL_RUNS") is not None, "Die Läufe-Zelle ist noch nicht vollständig durchgelaufen."
SUMMARY, VERDICT = {}, {}
for (L, e, n) in CANDIDATES:
    ctrl_D = [summarize(r)[0] for r in CTRL_RUNS[(L, e, n)]]
    for arm in ("frozen", "free"):
        est, lo, hi, sd = summarize(RUNS[(L, e, n, arm)])
        p = (1 + sum(c <= est for c in ctrl_D)) / (len(ctrl_D) + 1)
        n_pairs = len(RUNS[(L, e, n, arm)])
        mde = 2.8 * sd / math.sqrt(n_pairs)
        SUMMARY[(L, e, n, arm)] = dict(D=est, ci=(lo, hi), p_vs_controls=p, sd=sd, mde_80=mde, n_pairs=n_pairs,
                                        ctrl_median=sorted(ctrl_D)[len(ctrl_D) // 2])
        if lo <= 0 <= hi:
            v = "keine spezifische Wirkung (Intervall enthält 0)"
        elif est > 0:
            v = "erhöht die Ziel-Wörter"
        elif hi <= -0.05 and p <= 0.05:
            v = "spezifisch: Abnahme über die aktivitätsgleichen Kontrollen hinaus"
        elif hi <= -0.05:
            v = "Abnahme, aber nicht über alle Kontrollen hinaus (p > 0,05)"
        else:
            v = "unklar (Abnahme kleiner als 0,05 Nat)"
        VERDICT[(L, e, n, arm)] = v
        note = "  (Kontrollen: festgehaltenes Routing)" if arm == "free" else ""
        print(f"L{L}_E{e}_N{n} [{arm}]  D = {est:+.4f}  97,5-%-CI [{lo:+.4f}, {hi:+.4f}]  "
              f"p gegen Kontrollen = {p:.3f}  minimal nachweisbar (80 %) = {mde:.3f} Nat  -> {v}{note}")
    print(f"  Kontrollen L{L}_E{e}_N{n}: D-Median {sorted(ctrl_D)[len(ctrl_D) // 2]:+.4f}, Spanne {min(ctrl_D):+.4f} bis {max(ctrl_D):+.4f}")

In [ ]:
# ---------- Teil 3: Untergrenzen ----------
assert model is not None, "Dieser Teil braucht das Modell (P1_FROM_FILES = False)"
check_session(RC, STREAM)
remove_neuron_hooks(model)
RC.reset()
STREAM.ensure([p[0] for p in PAIRS])
NUM = {}
cand = CANDIDATES[0]
kl_floor = []
for kl_ids, _, _, _ in PAIRS[:5]:
    ids = torch.tensor(kl_ids, device=DEV)
    RC.mode = "free"
    with torch.no_grad():
        l1 = logits_of(model, ids[None])[0]
        lB = logits_of(model, ids[None].expand(BATCH, -1).contiguous())[0]
    kl_floor.append(float(stable_kl_f64(l1, lB).max()))
NUM["batch_size_floor_max_KL_nat"] = max(kl_floor)

hookN = NeuronHook(model, *cand, RC)
noop = []
for kl_ids, _, _, _ in PAIRS[:5]:
    tab = ablation_table(model, hookN, RC, torch.tensor(kl_ids, device=DEV), torch.tensor(kl_ids[1:] + [-1]),
                         W=W_WINDOW, batch=BATCH, scale=1.0)
    noop.append(float(tab["kl"].abs().max()))
hookN.remove()
NUM["noop_max_KL_nat"] = max(noop)

hookC2 = NeuronHook(model, *cand, RC)
neg_naive = tot = neg_stable = 0
for kl_ids, _, _, _ in PAIRS[:3]:
    T = len(kl_ids)
    ids = torch.tensor(kl_ids, device=DEV)
    full = ids[None].expand(BATCH, -1).contiguous()
    RC.mode = "free"
    with torch.no_grad():
        hookC2.mask, hookC2.scale = None, 0.0
        clean = logits_of(model, full)[0].float()
        m = torch.zeros(BATCH, T, dtype=torch.bool, device=DEV)
        for b in range(min(BATCH, T)):
            m[b, b] = True
        hookC2.mask, hookC2.scale = m, 0.0
        lg = logits_of(model, full)
        hookC2.mask = None
        for b in range(min(BATCH, T)):
            nk = naive_kl_f32(clean, lg[b].float())
            sk = stable_kl_f64(clean, lg[b].float())
            seg = slice(b, T)
            neg_naive += int((nk[seg] < 0).sum())
            neg_stable += int((sk[seg] < 0).sum())
            tot += T - b
hookC2.remove()
NUM["naive_f32_negative_values"] = neg_naive
NUM["stable_f64_negative_values"] = neg_stable
NUM["positions_checked"] = tot

pert = {}
for sc in (1 - 1e-4, 1 - 1e-2, 0.0):
    hookP = NeuronHook(model, *cand, RC)
    vals = []
    for kl_ids, _, _, _ in PAIRS[:5]:
        tab = ablation_table(model, hookP, RC, torch.tensor(kl_ids, device=DEV), torch.tensor(kl_ids[1:] + [-1]),
                             W=W_WINDOW, batch=BATCH, scale=sc)
        dg = tab["kl"].diagonal()
        vals.append({"max": float(dg.max()), "positions_nonzero": int((dg > 1e-12).sum()), "positions": int(dg.numel())})
    hookP.remove()
    pert[f"scale={sc:.6f}"] = vals
NUM["perturbation_at_own_position"] = pert
print(json.dumps(NUM, indent=1))

In [ ]:
# ---------- Ergebnis speichern und zusammenfassen ----------
def key_name(k):
    return f"L{k[0]}_E{k[1]}_N{k[2]}_{k[3]}"

out = {
    "config": dict(MODEL_ID=MODEL_ID, LIB_VERSION=LIB_VERSION, CANDIDATES=CANDIDATES, W_WINDOW=W_WINDOW, BATCH=BATCH, N_MATCH=N_MATCH,
                   CAL_ABS=CAL_ABS, CAL_REL=CAL_REL, GATE_VALUE=GATE_VALUE, GATE_ROUTE=GATE_ROUTE,
                   SMOKE=SMOKE, TEST_MODE=TEST_MODE, SEED=SEED, n_pairs=len(PAIRS)),
    "gate0_calibration_ok": globals().get("CALIB_OK"),
    "gate0": globals().get("GATE0_STATS"),
    "gate0b_amended": dict(used=globals().get("GATE0B", True), ok=globals().get("CALIB_B_OK"),
                           stats=globals().get("GATE0B_STATS"),
                           note="nachträglich festgelegt am 28.11.2026, Abweichung vom Plan"),
    "streaming": dict(SPLIT_LAYER=SPLIT_LAYER, STREAM_CHUNK=STREAM_CHUNK,
                      texts_prepared=len(STREAM.cache) if STREAM is not None else None,
                      lower_layer_loads=STREAM.stats["layer_loads"] if STREAM is not None else None),
    "P1_weights": P1,
    "P2_causal": {key_name(k): dict(D=round(v["D"], 5), ci97_5=[round(x, 5) for x in v["ci"]],
                                    p_vs_matched_controls=round(v["p_vs_controls"], 4), mde_80_nat=round(v["mde_80"], 4),
                                    verdict=VERDICT[k], n_pairs=v["n_pairs"])
                  for k, v in SUMMARY.items()},
    "P2_matched_control_D": {f"L{k[0]}_E{k[1]}_N{k[2]}": [round(summarize(r)[0], 5) for r in CTRL_RUNS[k]]
                             for k in CTRL_RUNS},
    "P3_numerics": NUM,
}
with open("kl_unembedding_results.json", "w", encoding="utf-8") as fh:
    json.dump(out, fh, indent=1, ensure_ascii=False)
print("Gespeichert: kl_unembedding_results.json")
print("Gate 0 (vorab festgelegt):", globals().get("CALIB_OK"), "| Gate 0b (nachträglich):", globals().get("CALIB_B_OK"))
for k, v in out["P2_causal"].items():
    print(k, "->", v["verdict"])

## Auslegung

- **P1a/P1b** beantworten die Frage nach einer *direkten* Ausgabe-Anomalie: Schreibt ein Neuron ungewöhnlich stark auf KL-Wörter, verglichen mit dem, was zufällige Wortgruppen in derselben Schicht bekommen?
- **P2** beantwortet die kausale Frage: Ändert das Ausschalten die Vorhersage von KL-Wörtern in KL-Texten mehr als in den Kontrolltexten, und mehr als bei Neuronen desselben Experten mit ähnlicher Aktivität?
- Ein Ergebnis „spezifisch“ in P2 ohne Auffälligkeit in P1 hieße: Die Wirkung läuft über spätere Schichten, nicht über den direkten Schreibvektor. Umgekehrt wäre eine starke direkte Wirkung ohne P2-Effekt ein Hinweis, dass das Modell das Wort über andere Wege erreicht.
- **P3** gibt die Messgenauigkeit an. Liegt das Rauschen der Batchgröße über dem minimal nachweisbaren Effekt aus P2, ist P2 nicht auswertbar.

Die Entscheidungen oben sind die vorab festgelegten Regeln. Sie ersetzen keine Interpretation; die Interpretation steht im Bericht.